# build_package
`silver_advisory_ghsa_affected`(`package_key`,`ecosystem`,`package_name` 재사용) → `silver_package`.
동일 패키지에 연결된 모든 취약점을 안정적으로 조회할 수 있도록 패키지를 하나의 Entity로 관리한다(02_Bronze_Silver_저장구조.md).
패키지 사실(ecosystem/name)은 한번 생기면 바뀌지 않으므로 신규 `package_key`만 append.

In [ ]:
%run 00_silver_common

In [ ]:
affected_df = spark.table(TBL["advisory_ghsa_affected"])

distinct_packages = (
    affected_df
    .filter(F.col("package_key").isNotNull())
    .select("package_key", "ecosystem", "package_name")
    .distinct()
    .withColumn("source", F.lit("ghsa"))
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingested_at", F.lit(INGESTED_AT))
)

if table_exists(TBL["package"]):
    existing_keys = spark.table(TBL["package"]).select("package_key")
    new_packages = distinct_packages.join(existing_keys, on="package_key", how="left_anti")
else:
    new_packages = distinct_packages

new_count = new_packages.count()
if new_count > 0:
    write_append(new_packages, TBL["package"])

log_batch_run("build_package", "SUCCESS", record_count=new_count)
print(f"[build_package] new_packages={new_count} batch_id={batch_id}")
